# Modernized DeepSORT — execution notebook

Thin orchestrator: every step calls the repo's scripts (logic lives in `eval/`, `data/`, `bodyreid/`). Runs top-to-bottom on **Colab Pro (GPU)**.

**Instructions:** set `REPO_URL`, Runtime → GPU, then Run all. Results are logged to `results/experiments.csv`; overlays to `overlays/`.


In [ ]:
# 1) Clone the repo and enter it (public repo -> no auth; if private, use a PAT URL)
REPO_URL = 'https://github.com/SergeySolovyev/Modernized-DeepSORT.git'
import os
if not os.path.isdir('Modernized-DeepSORT'):
    !git clone $REPO_URL Modernized-DeepSORT
%cd Modernized-DeepSORT


In [ ]:
# 2) Install dependencies
!pip -q install -r requirements-modern.txt
# torchreid (OSNet/ResNet50) is not on PyPI under this name -> install from source:
!pip -q install git+https://github.com/KaiyangZhou/deep-person-reid.git
!pip -q install -U openmim && mim install mmengine 'mmcv>=2.0' mmdet
!git clone -q https://github.com/JonathonLuiten/TrackEval third_party/TrackEval || true
!pip -q install -e third_party/TrackEval


In [ ]:
# 3) GPU check
!nvidia-smi


In [ ]:
# 4) Download + lay out MOT15/MOT16 (TrackEval structure)
!python -m data.download_mot


## Baseline — unmodified DeepSORT (provided detections + mars-small128)
Faithful baseline via the original legacy path. Requires `mars-small128.pb` at `third_party/deep_sort_data/` (see data/README.md).


In [ ]:
# 5) Baseline: unmodified DeepSORT (provided detections + mars-small128).
# Requires third_party/deep_sort_data/mars-small128.pb (see data/README.md).
!python -m eval.run_baseline --mars third_party/deep_sort_data/mars-small128.pb
!python -m eval.trackeval_runner --benchmark MOT15 --trackers baseline
!python -m eval.trackeval_runner --benchmark MOT16 --trackers baseline


## Detector study — Precision / Recall / F1 vs GT (IoU≥0.5)


In [ ]:
for det in ['yolo', 'nanodet', 'mmdet']:
    !python -m eval.det_eval --detector $det --device cuda


## REID study — REID-only HOTA (GT boxes, SORT detection disabled)


In [ ]:
!python -m eval.reid_eval --reids osnet_x1_0 osnet_ain_x1_0 resnet50 timm_mobilenet mars --device cuda


## Full pipeline — best combo, live tracking → HOTA


In [ ]:
DET, REID = 'yolo', 'osnet_x1_0'
!python -m eval.run_tracking --detector $DET --reid $REID --device cuda
!python -m eval.trackeval_runner --benchmark MOT15 --trackers ${DET}__${REID}
!python -m eval.trackeval_runner --benchmark MOT16 --trackers ${DET}__${REID}


In [ ]:
# FPS (must be >= 5 FPS for the real-time requirement)
!python -m eval.fps_bench --detector $DET --reid $REID --sequence MOT16-09 --device cuda


## Additional task — standalone body-REID identity system


In [ ]:
# Standalone REID model/param selection on GT crops
!python -m data.prepare_gt_crops
!python -m bodyreid.eval.extract_gt --reid $REID --out descriptors_$REID.npz --device cuda
!python -m bodyreid.eval.cluster_eval --npz descriptors_$REID.npz
!python -m bodyreid.eval.sweep --npz descriptors_$REID.npz --reid $REID


In [ ]:
# Full pipeline WITH the identity system
!python -m eval.run_tracking --detector $DET --reid $REID --bodyreid --device cuda


## Segmentation — YOLOv8-seg (mask→bbox)


In [ ]:
!python -m eval.det_eval --detector yolo_seg --device cuda
!python -m eval.run_tracking --detector yolo_seg --reid $REID --device cuda


## Overlays — baseline vs best


In [ ]:
import os; os.makedirs('overlays', exist_ok=True)
!python -m eval.make_overlays --detector gt   --reid mars  --sequence MOT16-09 --mode gtbox --out overlays/baseline_MOT16-09.mp4 --device cuda
!python -m eval.make_overlays --detector $DET --reid $REID --sequence MOT16-09 --out overlays/best_MOT16-09.mp4 --device cuda


## Results summary


In [ ]:
# Auto-build the per-video HOTA table (tracker x video + Mean, Delta vs baseline)
!python -m eval.summarize
import pandas as pd
print(open('report/results_table.md').read())
df = pd.read_csv('results/experiments.csv')
df.tail(40)
